# Fine-Tune YOLO26 - Image Classification

**Goal.** Fine-tune **YOLO26-nano-classification** on the Rock-Paper-Scissors dataset. Practical, runnable, comment-rich. For the *concepts* behind every choice, see `yolo_img_classification.ipynb`.

**What changed:**
- Added context for every argument (`patience`, `imgsz`, `batch`, `seed`, etc.).
- Pinned a reproducible run name so you don't end up with `train/`, `train2/`, `train3/` in your outputs folder.
- Added a quick post-training validation + plot inspection.
- Added an `imgsz` upgrade note (the original used `150` to match the prior Keras notebook — for transfer learning, `224` is generally better).


## 1. Install & sanity check


In [1]:
# uv add ultralytics

import ultralytics
from ultralytics import YOLO

ultralytics.checks()

Ultralytics 8.4.46 🚀 Python-3.11.14 torch-2.2.2 CPU (Intel Core i7-9750H 2.60GHz)
Setup complete ✅ (12 CPUs, 16.0 GB RAM, 194.8/465.6 GB disk)


## 2. Load the pretrained backbone

`yolo26n-cls.pt` ships pretrained on ImageNet. Ultralytics will download it automatically on first call (~6 MB).

When `model.train()` runs with a 3-class dataset, the final classification layer is **automatically replaced** to output 3 logits — you don't have to do anything.


In [2]:
model = YOLO('yolo26n-cls.pt')    # nano = smallest, fastest variant
model.info()                      # parameter count + FLOPs

YOLO26n-cls summary: 120 layers, 2,812,104 parameters, 0 gradients, 4.3 GFLOPs


(120, 2812104, 0, 4.2763264)

## 3. Train

**Why these arguments?**

- `data='rps_dataset'` — folder with `train/` and `val/` subdirs (each containing `paper/`, `rock/`, `scissors/`).
- `epochs=20` — RPS is small and pretraining is strong; 20 epochs is plenty. Early stopping will catch overfitting earlier if it appears.
- `imgsz=224` — YOLO classification's pretrained default. Changing this requires re-cropping the pretrained features. *We recommend 224 over the original 150.* (Keep 150 only if matching the Keras CNN's resolution is important to you.)
- `patience=5` — stop if validation top-1 accuracy doesn't improve for 5 consecutive epochs.
- `seed=0` — reproducibility.
- `name='rps_yolo26n'` — output folder will be `runs/classify/rps_yolo26n/` (instead of the auto-incremented `train`, `train2`, ...).


In [3]:
from pathlib import Path
PROJECT_DIR = Path.cwd() / 'runs' / 'classify'

In [4]:
results = model.train(
    data     = 'rps_dataset',
    epochs   = 20,
    imgsz    = 224,            # use 150 only if you must match the Keras CNN
    batch    = 32,
    patience = 5,
    seed     = 0,
    project  = str(PROJECT_DIR),
    name     = 'rps_yolo26n',
    exist_ok = True,           # overwrite the run folder if you re-run this cell
    plots    = True,
)

Ultralytics 8.4.46 🚀 Python-3.11.14 torch-2.2.2 CPU (Intel Core i7-9750H 2.60GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=rps_dataset, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rps_yolo26n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=5, perspective=

## 4. Where did everything go?

Outputs land in `runs/classify/rps_yolo26n/`. The two files you'll use most:

- `weights/best.pt` — best-validation checkpoint. **Use this for inference.**
- `results.png` — accuracy / loss curves. **Always look at this.**

Open `train_batch0.jpg` to see real augmented training images. If the augmentation is too aggressive (gestures unrecognisable), reduce it via train args; if too mild, increase `mixup` or `erasing`.


In [5]:
from pathlib import Path
run_dir = Path('runs/classify/rps_yolo26n')
print('contents of run dir:')
for p in sorted(run_dir.iterdir()):
    print(' ', p.name)

contents of run dir:
  args.yaml
  confusion_matrix.png
  confusion_matrix_normalized.png
  results.csv
  results.png
  train_batch0.jpg
  train_batch1.jpg
  train_batch2.jpg
  val_batch0_labels.jpg
  val_batch0_pred.jpg
  val_batch1_labels.jpg
  val_batch1_pred.jpg
  val_batch2_labels.jpg
  val_batch2_pred.jpg
  weights


## 5. Quick validation

Confirms the saved weights match what you expect.


In [6]:
metrics = model.val()
print(f'top-1: {metrics.top1:.4f}')
print(f'top-5: {metrics.top5:.4f}')   # always 1.0 for our 3-class problem

Ultralytics 8.4.46 🚀 Python-3.11.14 torch-2.2.2 CPU (Intel Core i7-9750H 2.60GHz)
YOLO26n-cls summary (fused): 47 layers, 1,529,867 parameters, 0 gradients, 3.2 GFLOPs
WARNING ⚠️ Dataset 'split=val' not found, using 'split=test' instead.
train: /Users/mac/Desktop/Apr 2026/AI/AI-Afternoon/dl-project/notebooks/rps_dataset/train... found 2520 images in 3 classes ✅ 
val: /Users/mac/Desktop/Apr 2026/AI/AI-Afternoon/dl-project/notebooks/rps_dataset/test... found 372 images in 3 classes ✅ 
test: /Users/mac/Desktop/Apr 2026/AI/AI-Afternoon/dl-project/notebooks/rps_dataset/test... found 372 images in 3 classes ✅ 
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 955.4±241.4 MB/s, size: 82.7 KB)
val: Scanning /Users/mac/Desktop/Apr 2026/AI/AI-Afternoon/dl-project/notebooks/rps_dataset/test... 372 images, 0 corrupt: 100% ━━━━━━━━━━━━ 372/372 78.0Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 24/24 5.8it/s 4.1s0.2s
                   all          1          1
Speed: 0.0

## 6. Display the training curves inline


In [7]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open('runs/classify/rps_yolo26n/results.png')
plt.figure(figsize=(12, 6))
plt.imshow(img)
plt.axis('off')
plt.show()

<Figure size 1200x600 with 1 Axes>

## 7. Next step

Open **`test-yolo.ipynb`** to load `runs/classify/rps_yolo26n/weights/best.pt` and run inference on new images.

Try one experiment to deepen intuition:
1. Re-run with `imgsz=150` and compare final accuracy / training time.
2. Re-run with the medium variant (`yolo26m-cls.pt`) — should be 2-3 absolute points more accurate, ~3× slower.
